In [1]:
from transformers import (
  AutoModelForSequenceClassification,
  AutoModelForTokenClassification,
  AutoTokenizer,
  Trainer,
  TrainingArguments,
  pipeline,
)
import mlflow
import torch
import os

In [2]:
torch.__version__

'2.10.0+cu128'

In [ ]:
os.environ['MLFLOW_S3_ENDPOINT_URL'] = "http://localhost:9000" # Cổng MinIO bạn đã sửa ở bước trước
os.environ['AWS_ACCESS_KEY_ID'] = "minioadmin"                # Mặc định của MinIO
os.environ['AWS_SECRET_ACCESS_KEY'] = "minioadmin123"

In [4]:
mlflow.set_tracking_uri("http://localhost:5000")

In [5]:
client = mlflow.MlflowClient(tracking_uri="http://localhost:5000")
experiments = client.search_experiments()
print(f"\n✅ MLflow connection successful!")
print(f"   Total experiments: {len(experiments)}")


✅ MLflow connection successful!
   Total experiments: 3


In [6]:
mlflow.set_experiment("phobert-ner")

<Experiment: artifact_location='s3://mlflow-artifacts/1', creation_time=1768466325864, experiment_id='1', last_update_time=1768466325864, lifecycle_stage='active', name='phobert-ner', tags={}>

In [7]:
model = AutoModelForTokenClassification.from_pretrained("/home/anjuly/Desktop/AIDE1/models/phobert-ner-model")

In [10]:
tokenizer = AutoTokenizer.from_pretrained("/home/anjuly/Desktop/AIDE1/models/phobert-ner-model")

In [11]:
run_id = "8ceaf368dab84d229fb11de54dad7c9e"

In [12]:
tuned_pipeline = pipeline(
  task="ner",
  model=model,
  batch_size=8,
  tokenizer=tokenizer,
  device="cpu",
)

Device set to use cpu


In [13]:
quick_check = (
    "Bệnh nhân Nguyễn Văn A, 35 tuổi, trú tại Cầu Giấy, Hà Nội. "
    "Nhập viện ngày 20/01/2026 với triệu chứng sốt cao và ho kéo dài."
)
result = tuned_pipeline(quick_check)
result

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity': 'B-NAME',
  'score': np.float32(0.8524954),
  'index': 3,
  'word': 'Nguyễn',
  'start': None,
  'end': None},
 {'entity': 'I-NAME',
  'score': np.float32(0.8089365),
  'index': 4,
  'word': 'Văn',
  'start': None,
  'end': None},
 {'entity': 'I-NAME',
  'score': np.float32(0.81444514),
  'index': 5,
  'word': 'A@@',
  'start': None,
  'end': None},
 {'entity': 'B-AGE',
  'score': np.float32(0.86551046),
  'index': 7,
  'word': '35',
  'start': None,
  'end': None},
 {'entity': 'B-LOCATION',
  'score': np.float32(0.97446525),
  'index': 13,
  'word': 'Cầu',
  'start': None,
  'end': None},
 {'entity': 'I-LOCATION',
  'score': np.float32(0.9763949),
  'index': 14,
  'word': 'Giấ@@',
  'start': None,
  'end': None},
 {'entity': 'I-LOCATION',
  'score': np.float32(0.975024),
  'index': 15,
  'word': 'y@@',
  'start': None,
  'end': None},
 {'entity': 'B-LOCATION',
  'score': np.float32(0.974603),
  'index': 17,
  'word': 'Hà',
  'start': None,
  'end': None},
 {'entity': 'I-LO

In [14]:
model_config = {"batch_size": 8}
signature = mlflow.models.infer_signature(
    [quick_check],
    mlflow.transformers.generate_signature_output(tuned_pipeline, [quick_check]),
    params=model_config,
)

In [ ]:
with mlflow.start_run(run_id=run_id) :
    model_info = mlflow.transformers.log_model(
        transformers_model=tuned_pipeline,
        name="fine_tuned",
        signature=signature,
        input_example=[quick_check],
        model_config=model_config,
    )

2026/01/25 23:38:14 WARNING mlflow.transformers: The model card could not be retrieved from the hub due to Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/home/anjuly/Desktop/AIDE1/models/phobert-ner-model'. Use `repo_type` argument if needed.
2026/01/25 23:38:14 WARNING mlflow.transformers: Unable to find license information for this model. Please verify permissible usage for the model you are storing prior to use.
2026/01/25 23:38:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cpu
2026/01/25 23:38:19 WARNING mlflow.transformers: params provided to the `predict` method will override the inference configuration saved with the model. If the params provided are not valid for the pipeline, MlflowException will be raised.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


🏃 View run awesome-dog-323 at: http://localhost:5000/#/experiments/1/runs/8ceaf368dab84d229fb11de54dad7c9e
🧪 View experiment at: http://localhost:5000/#/experiments/1


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/anjuly/Desktop/AIDE1/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/anjuly/Desktop/AIDE1/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py", line 341, in dispatch_control
    await self.process_control(msg)
  File "/home/anjuly/Desktop/AIDE1/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py", line 347, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/anjuly/Desktop/AIDE1/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/anjuly/Desktop/AIDE1/.venv/lib/python3.11/site-pac